# Optional Lab B: Inverse Dynamics for 8-Direction Center-Out Reaching**Computational Sensorimotor Control — Week 6 Supplement**In the lecture and lab, we applied the inverse dynamics pipeline to a single semicircular arc. This tutorial extends the pipeline to the classic 8-direction center-out reaching task from Weeks 4–5, showing how torque requirements and optimal muscle activations change with reach direction.**No TODOs — this is a walkthrough.** Run each cell and study the outputs.

In [ ]:
!pip3 install git+https://github.com/tarkeshsingh/computational-sensorimotor-control.git#subdirectory=smc -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize as scipy_minimize
plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 11,
                      'axes.grid': True, 'grid.alpha': 0.3})

from smc import (
    Arm, make_muscles, simulate_lambda,
    Q_REF, MUSCLE_DEFS,
    mass_matrix, coriolis, joint_accelerations, rk4_step,
)

arm = Arm()
ik  = arm.inverse_kinematics
fk  = arm.forward_kinematics
start_hand = fk(Q_REF)

NAVY='#1B2A4A'; TEAL='#2E86AB'; GREEN='#27AE60'; RED='#E74C3C'
ORANGE='#E67E22'; PURPLE='#8E44AD'
R_mat = np.array([[m[3] for m in MUSCLE_DEFS], [m[4] for m in MUSCLE_DEFS]])
rho = np.array([m[1] for m in MUSCLE_DEFS])
mnames = ['Pec', 'BicL', 'BicS', 'Delt', 'TriL', 'TriLg']
mcols = [RED, ORANGE, '#E8735A', TEAL, '#3498db', GREEN]

print(f'Start: ({start_hand[0]*100:.1f}, {start_hand[1]*100:.1f}) cm')
print(f'Q_REF = ({np.degrees(Q_REF[0]):.0f}°, {np.degrees(Q_REF[1]):.0f}°)')

---## Part 1: Define the 8 Targets and Generate TrajectoriesWe place 8 targets at 8 cm from Q_REF, evenly spaced at 0°, 45°, 90°, ..., 315°. For each direction, we generate a minimum-jerk straight-line trajectory in 600 ms.

In [ ]:
REACH_DIST = 0.08  # 8 cm
T_REACH = 0.600    # 600 ms
dt = 0.001
DIRECTIONS = np.arange(0, 360, 45)  # 0°, 45°, ..., 315°

# Min-jerk basis
def minjerk_straight(start, end, T, dt=0.001):
    t = np.arange(0, T + dt, dt)
    tau = t / T
    s = 10*tau**3 - 15*tau**4 + 6*tau**5
    pos = start[None, :] + s[:, None] * (end - start)[None, :]
    return t, pos

# Generate targets and trajectories
targets = {}
trajectories = {}

plt.figure(figsize=(6, 6))
plt.plot(start_hand[0]*100, start_hand[1]*100, 'ko', ms=10, zorder=5, label='Start')

reachable_dirs = []
for deg in DIRECTIONS:
    rad = np.radians(deg)
    target = start_hand + REACH_DIST * np.array([np.cos(rad), np.sin(rad)])
    try:
        q_t = ik(target[0], target[1])
        if not arm.in_joint_limits(q_t):
            print(f'{deg:3d}°: target out of joint limits')
            continue
    except:
        print(f'{deg:3d}°: IK failed')
        continue
    targets[deg] = target
    t, pos = minjerk_straight(start_hand, target, T_REACH, dt)
    trajectories[deg] = (t, pos)
    reachable_dirs.append(deg)
    plt.plot(target[0]*100, target[1]*100, 'r*', ms=12, zorder=5)
    plt.annotate(f'{deg}°', (target[0]*100+0.3, target[1]*100+0.3), fontsize=9)
    plt.plot(pos[:,0]*100, pos[:,1]*100, 'gray', lw=1, alpha=0.5)

plt.xlabel('X (cm)'); plt.ylabel('Y (cm)')
plt.title('8-direction center-out targets (8 cm from Q_REF)')
plt.legend(); plt.axis('equal'); plt.show()
print(f'Reachable directions: {reachable_dirs}')

---## Part 2: The Inverse Dynamics Pipeline for Each DirectionFor each reachable direction, apply the full ID pipeline:1. IK at every timestep: P(t) → q(t)2. Differentiate: q(t) → q̇(t), q̈(t)3. Inverse dynamics: τ(t) = M(q)·q̈ + C(q, q̇)

In [ ]:
id_results = {}

for deg in reachable_dirs:
    t, pos = trajectories[deg]
    n = len(t)
    
    # Step 1: IK
    q = np.zeros((n, 2))
    for i in range(n):
        q[i] = ik(pos[i, 0], pos[i, 1])
    
    # Step 2: Differentiate
    qd = np.zeros_like(q); qdd = np.zeros_like(q)
    qd[1:-1] = (q[2:] - q[:-2]) / (2*dt); qd[0] = qd[1]; qd[-1] = qd[-2]
    qdd[1:-1] = (q[2:] - 2*q[1:-1] + q[:-2]) / dt**2; qdd[0] = qdd[1]; qdd[-1] = qdd[-2]
    
    # Step 3: Inverse dynamics
    tau = np.zeros((n, 2))
    tau_M = np.zeros((n, 2)); tau_C = np.zeros((n, 2))
    for i in range(n):
        M = mass_matrix(q[i]); C = coriolis(q[i], qd[i])
        tau_M[i] = M @ qdd[i]; tau_C[i] = C
        tau[i] = tau_M[i] + C
    
    id_results[deg] = dict(t=t, pos=pos, q=q, qd=qd, qdd=qdd,
                           tau=tau, tau_M=tau_M, tau_C=tau_C)

print(f'ID computed for {len(id_results)} directions.')

---## Part 3: Torque Profiles Across DirectionsPlot shoulder and elbow torques for all 8 directions, color-coded. This shows how the required torque pattern changes with reach direction.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

cmap = plt.cm.hsv
for i, deg in enumerate(reachable_dirs):
    r = id_results[deg]
    color = cmap(i / len(reachable_dirs))
    ax1.plot(r['t']*1000, r['tau'][:,0], color=color, lw=1.5, label=f'{deg}°')
    ax2.plot(r['t']*1000, r['tau'][:,1], color=color, lw=1.5, label=f'{deg}°')

ax1.set_xlabel('Time (ms)'); ax1.set_ylabel('Torque (N·m)')
ax1.set_title('A. Shoulder torque'); ax1.legend(fontsize=7, ncol=2)

ax2.set_xlabel('Time (ms)'); ax2.set_ylabel('Torque (N·m)')
ax2.set_title('B. Elbow torque'); ax2.legend(fontsize=7, ncol=2)

plt.suptitle('Required joint torques for 8-direction center-out reaching', fontweight='bold')
plt.tight_layout(); plt.show()

Notice how:- The shoulder torque sign flips between leftward (positive, flexion) and rightward (negative, extension) reaches- The elbow torque depends on the combined direction: diagonal reaches produce the largest elbow torques because both joints must contribute- Every direction has a **biphasic** profile: accelerate then decelerate

---## Part 4: Static Optimization — Muscle Forces for Each DirectionApply the Crowninshield criterion (n = 3) at every timestep for every direction.

In [ ]:
def static_opt(tau_req, n_power=3):
    def cost(F): return np.sum((F / rho) ** n_power)
    def cost_jac(F): return n_power * (F / rho) ** (n_power - 1) / rho
    constraints = {'type': 'eq', 'fun': lambda F: R_mat @ F - tau_req, 'jac': lambda F: R_mat}
    res = scipy_minimize(cost, np.ones(6)*0.1, jac=cost_jac, method='SLSQP',
                         bounds=[(0,None)]*6, constraints=constraints,
                         options={'maxiter': 500, 'ftol': 1e-12})
    return res.x if res.success else np.zeros(6)

# Compute optimal forces for all directions
F_results = {}
for deg in reachable_dirs:
    r = id_results[deg]
    n = len(r['t'])
    F_all = np.zeros((n, 6))
    for i in range(n):
        if np.linalg.norm(r['tau'][i]) > 1e-6:
            F_all[i] = static_opt(r['tau'][i])
    F_results[deg] = F_all
    print(f'{deg:3d}°: peak forces = {[f"{F_all[:,j].max():.1f}" for j in range(6)]}')

---## Part 5: Direction Tuning — Which Muscles Prefer Which Directions?For each muscle and each direction, extract the peak activation. This produces a direction-tuning curve analogous to what Georgopoulos et al. (1982) observed in motor cortex neurons.

In [ ]:
# Peak activation per muscle per direction
peak_matrix = np.zeros((6, len(reachable_dirs)))
for j, deg in enumerate(reachable_dirs):
    for m in range(6):
        peak_matrix[m, j] = F_results[deg][:, m].max()

# Normalize each muscle by its global max
peak_norm = peak_matrix.copy()
for m in range(6):
    mx = peak_matrix[m].max()
    if mx > 0: peak_norm[m] /= mx

# Heatmap
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(peak_norm, aspect='auto', cmap='inferno', vmin=0, vmax=1)
ax.set_xticks(range(len(reachable_dirs)))
ax.set_xticklabels([f'{d}°' for d in reachable_dirs])
ax.set_yticks(range(6)); ax.set_yticklabels(mnames)
ax.set_xlabel('Reach direction'); ax.set_ylabel('Muscle')
ax.set_title('Direction tuning: normalized peak muscle force')
plt.colorbar(im, ax=ax, label='Normalized peak force')

# Annotate
for m in range(6):
    for j in range(len(reachable_dirs)):
        val = peak_norm[m, j]
        color = 'white' if val < 0.5 else 'black'
        ax.text(j, m, f'{val:.2f}', ha='center', va='center', fontsize=8, color=color)

plt.tight_layout(); plt.show()

In [ ]:
# Polar tuning curves for each muscle
fig, axes = plt.subplots(2, 3, figsize=(14, 9), subplot_kw=dict(projection='polar'))

for m, ax in enumerate(axes.flat):
    dirs_rad = np.radians(reachable_dirs)
    # Close the curve
    vals = np.append(peak_norm[m], peak_norm[m, 0])
    angles = np.append(dirs_rad, dirs_rad[0])
    ax.fill(angles, vals, color=mcols[m], alpha=0.3)
    ax.plot(angles, vals, color=mcols[m], lw=2)
    ax.set_title(mnames[m], fontweight='bold', color=mcols[m], pad=15)
    ax.set_ylim(0, 1.1)
    ax.set_thetagrids(reachable_dirs)

plt.suptitle('Muscle direction tuning curves (normalized peak force)', fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

print('Each muscle has a preferred direction determined by its moment arm geometry.')
print('Flexors (Pec, BicL, BicS) prefer directions that require positive torque.')
print('Extensors (Delt, TriL, TriLg) prefer the opposite directions.')

---## Part 6: Full Trajectory Activations — Two Example DirectionsShow the complete time-course of muscle activations for a rightward (0°) and upward (90°) reach side by side.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, deg in enumerate([0, 90]):
    if deg not in F_results:
        print(f'{deg}° not reachable, skipping')
        continue
    ax = axes[idx]
    for j in range(6):
        ax.plot(id_results[deg]['t']*1000, F_results[deg][:, j],
                color=mcols[j], lw=1.5, label=mnames[j])
    ax.set_xlabel('Time (ms)'); ax.set_ylabel('Force (N)')
    ax.set_title(f'{deg}° reach', fontweight='bold')
    ax.legend(fontsize=7, ncol=2)

plt.suptitle('Optimal muscle forces for two reach directions (n=3, 600 ms, 8 cm)', fontweight='bold')
plt.tight_layout(); plt.show()

print('0° (rightward): extensors dominate — Delt pushes the hand rightward')
print('90° (upward): flexors and biarticulars contribute — requires both shoulder and elbow flexion')

---## Summary1. **The inverse dynamics pipeline generalizes** to any trajectory — straight reaches, curved arcs, or arbitrary paths. The same 3-step process (IK → differentiate → τ = Mq̈ + C) works for all of them.2. **Torque profiles are direction-dependent.** The biphasic (accelerate/decelerate) shape is universal, but the sign, magnitude, and relative contribution of shoulder vs elbow vary with direction.3. **Each muscle has a preferred direction** determined by its moment arm geometry. This is a direct prediction of the inverse dynamics framework — the optimizer selects muscles whose moment arms align with the required torque.4. **Direction tuning in muscles parallels direction tuning in motor cortex** (Georgopoulos et al. 1982). Whether the brain computes this through inverse dynamics + optimization, or through a different mechanism (like threshold control), is one of the central questions we explore in Weeks 7–11.